# Riset Klasifikasi Kesesuaian Tanaman Berdasarkan Kadar Nutrisi Tanah (N, P, K, pH) Menggunakan Optimasi XGBoost
**Sumber Dataset:** [Kaggle - Crop Recommendation Dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)

Notebook ini mencakup:
1. **Exploratory Data Analysis (EDA)** kadar makronutrisi tanah (`N`, `P`, `K`) dan keasaman (`ph`) pada 22 komoditas tanaman.
2. **Agronomic Feature Engineering** (rasio keseimbangan `N/P`, `N/K`, `P/K`, proporsi NPK, dan deviasi pH netral).
3. **Ablation Study** (membandingkan model dengan fitur kadar tanah murni vs fitur tanah + rasio vs gabungan agroklimat).
4. **Komparasi Model & Hyperparameter Tuning XGBoost** menggunakan 5-Fold Stratified Cross-Validation.
5. **Analisis Feature Importance & SHAP Explainable AI**.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
import os, sys
sys.path.append(os.path.abspath('..'))
df = pd.read_csv('../data/Crop_recommendation.csv' if os.path.exists('../data/Crop_recommendation.csv') else 'data/Crop_recommendation.csv')
print('Ukuran Dataset:', df.shape)
df.head()

In [ ]:
# Feature Engineering Rasio Kandungan Tanah
from riset_tanah_tanaman_xgboost import add_soil_agronomic_features, run_ablation_study, train_and_evaluate_models

df_feat = add_soil_agronomic_features(df)
le = LabelEncoder()
y_all = le.fit_transform(df_feat['label'])
X_all = df_feat.drop(columns=['label'])

ablation_df = run_ablation_study(df_feat, y_all)
ablation_df

In [ ]:
# Pelatihan & Evaluasi Model XGBoost
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=42
)
best_xgb, comparison_df, y_pred = train_and_evaluate_models(X_train, X_test, y_train, y_test, le)
comparison_df